# 04 · LLM Model Internals [SOLUTION]

> **Focus**: VRAM calculations, quantization impact, inference profiling

This notebook explores the practical constraints of running LLMs:
1. **VRAM Calculator** — determine memory requirements for different model configs
2. **Quantization Benchmark** — measure accuracy/memory/speed trade-offs
3. **Inference Profiling** — identify bottlenecks in forward pass

**Requirements**: GPU with ≥8 GB VRAM, PyTorch, transformers, bitsandbytes

## 0 · Environment Setup

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Verify GPU availability
if not torch.cuda.is_available():
    print("⚠ Warning: No GPU detected — some exercises will be skipped")
    print("  Exercise 1 (VRAM calculator) can run on CPU")
    print("  Exercises 2-3 require GPU")
else:
    device = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {device}")
    print(f"  VRAM: {vram:.1f} GB")

## Exercise 1 — VRAM Calculator

**Goal**: Calculate memory requirements for different model configurations.

**Memory components**:
- **Model weights**: `params × bytes_per_param`
- **Activations**: `batch_size × seq_len × hidden_dim × num_layers × bytes_per_activation`
- **KV Cache**: `2 × batch_size × seq_len × num_layers × num_heads × head_dim × bytes_per_param`

**Precision mapping**:
- fp32: 4 bytes
- fp16/bf16: 2 bytes
- int8: 1 byte
- int4: 0.5 bytes

In [ ]:
def calculate_vram(
    params: int,           # Total parameters (e.g., 7e9 for 7B)
    precision: str,        # 'fp32', 'fp16', 'int8', 'int4'
    seq_len: int,          # Sequence length
    batch_size: int,       # Batch size
    hidden_dim: int = 4096,      # Model hidden dimension
    num_layers: int = 32,        # Number of transformer layers
    num_heads: int = 32,         # Number of attention heads
) -> dict:
    """
    Calculate VRAM requirements for LLM inference.

    Returns:
        dict with keys: weights_gb, activations_gb, kv_cache_gb, total_gb
    """
    # Map precision to bytes per parameter
    precision_map = {
        'fp32': 4,
        'fp16': 2,
        'bf16': 2,
        'int8': 1,
        'int4': 0.5,
    }
    bytes_per_param = precision_map[precision]

    # Calculate weights memory
    weights_bytes = params * bytes_per_param
    weights_gb = weights_bytes / 1e9

    # Calculate activations memory (rough estimate)
    # Each layer holds intermediate activations of size (batch × seq × hidden)
    # We need to store them for all layers during forward pass
    activations_bytes = batch_size * seq_len * hidden_dim * num_layers * 2  # fp16 for activations
    activations_gb = activations_bytes / 1e9

    # Calculate KV cache memory
    # For each layer, we store K and V tensors: (batch × seq × num_heads × head_dim)
    head_dim = hidden_dim // num_heads
    kv_cache_bytes = 2 * batch_size * seq_len * num_layers * num_heads * head_dim * bytes_per_param
    kv_cache_gb = kv_cache_bytes / 1e9

    total_gb = weights_gb + activations_gb + kv_cache_gb

    return {
        'weights_gb': round(weights_gb, 2),
        'activations_gb': round(activations_gb, 2),
        'kv_cache_gb': round(kv_cache_gb, 2),
        'total_gb': round(total_gb, 2),
    }

# Test with a simple example
result = calculate_vram(
    params=7_000_000_000,  # 7B
    precision='fp16',
    seq_len=2048,
    batch_size=1,
    hidden_dim=4096,
    num_layers=32,
    num_heads=32,
)
print("LLaMA 7B @ fp16, seq_len=2048, batch=1:")
print(f"  Weights: {result['weights_gb']} GB")
print(f"  Activations: {result['activations_gb']} GB")
print(f"  KV Cache: {result['kv_cache_gb']} GB")
print(f"  Total: {result['total_gb']} GB")

### Test on LLaMA configurations

In [ ]:
# LLaMA model configurations
configs = [
    # name, params, hidden_dim, num_layers, num_heads
    ('LLaMA-7B', 7_000_000_000, 4096, 32, 32),
    ('LLaMA-13B', 13_000_000_000, 5120, 40, 40),
    ('LLaMA-70B', 70_000_000_000, 8192, 80, 64),
]

precisions = ['fp16', 'int8', 'int4']
seq_lengths = [512, 2048]
batch_size = 1

results = []
for model_name, params, hidden_dim, num_layers, num_heads in configs:
    for precision in precisions:
        for seq_len in seq_lengths:
            vram = calculate_vram(
                params=params,
                precision=precision,
                seq_len=seq_len,
                batch_size=batch_size,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                num_heads=num_heads,
            )
            results.append({
                'model': model_name,
                'precision': precision,
                'seq_len': seq_len,
                'weights_gb': vram['weights_gb'],
                'activations_gb': vram['activations_gb'],
                'kv_cache_gb': vram['kv_cache_gb'],
                'total_gb': vram['total_gb'],
            })

df = pd.DataFrame(results)
print("\nVRAM Requirements for LLaMA Models:")
print(df.to_string(index=False))

### Can it fit on RTX 4090?

In [ ]:
# RTX 4090 has 24 GB VRAM
RTX_4090_VRAM = 24.0

df['fits_rtx_4090'] = df['total_gb'] <= RTX_4090_VRAM

print(f"\nConfigurations that fit on RTX 4090 ({RTX_4090_VRAM} GB):")
print(df[['model', 'precision', 'seq_len', 'total_gb', 'fits_rtx_4090']].to_string(index=False))

print("\n📊 Summary:")
fits_count = df['fits_rtx_4090'].sum()
total_count = len(df)
print(f"  {fits_count}/{total_count} configurations fit on RTX 4090")
print(f"\n  ✓ All 7B and 13B models fit in int4")
print(f"  ✓ 7B fits in fp16 at short sequences")
print(f"  ✗ 70B requires multi-GPU even in int4 at long sequences")

## Exercise 2 — Quantization Impact Benchmark

**Goal**: Compare model quality, memory usage, and speed across quantization levels.

**Setup**: Load same model (GPT-2 or similar) in fp16, int8, int4.

**Note**: This exercise requires GPU. Skip if CUDA not available.

In [ ]:
if not torch.cuda.is_available():
    print("⚠ Skipping Exercise 2 — GPU required")
else:
    print("✓ GPU available — proceeding with quantization benchmark")

### Load model in different precisions

In [ ]:
if torch.cuda.is_available():
    model_name = "gpt2"  # Using GPT-2 as it's small and widely available
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    models = {}
    memory_usage = {}

    # Load fp16 model
    print("Loading fp16 model...")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    models['fp16'] = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    memory_usage['fp16'] = torch.cuda.max_memory_allocated() / 1e9
    print(f"  fp16 memory: {memory_usage['fp16']:.2f} GB")

    # Load int8 model
    print("\nLoading int8 model...")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    models['int8'] = AutoModelForCausalLM.from_pretrained(
        model_name,
        load_in_8bit=True,
        device_map="auto",
    )
    memory_usage['int8'] = torch.cuda.max_memory_allocated() / 1e9
    print(f"  int8 memory: {memory_usage['int8']:.2f} GB")

    # Load int4 model
    print("\nLoading int4 model...")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    models['int4'] = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    memory_usage['int4'] = torch.cuda.max_memory_allocated() / 1e9
    print(f"  int4 memory: {memory_usage['int4']:.2f} GB")

    print("\n✓ All models loaded")

### Run benchmark on sample prompts

In [ ]:
if torch.cuda.is_available():
    benchmark_prompts = [
        "The capital of France is",
        "To calculate the area of a circle, you need",
        "In 1969, humans landed on",
        "The speed of light is approximately",
        "Python is a programming language that",
    ]

    results = []

    for precision, model in models.items():
        print(f"\nBenchmarking {precision}...")
        for i, prompt in enumerate(benchmark_prompts):
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            # Warmup run
            with torch.no_grad():
                _ = model.generate(**inputs, max_new_tokens=20, do_sample=False)

            # Timed run
            start = time.perf_counter()
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)
            end = time.perf_counter()

            generation_time_ms = (end - start) * 1000
            output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            results.append({
                'precision': precision,
                'prompt_id': i,
                'prompt': prompt,
                'generation_time_ms': round(generation_time_ms, 2),
                'output': output_text,
            })

            print(f"  Prompt {i}: {generation_time_ms:.2f} ms")

    benchmark_df = pd.DataFrame(results)
    print("\n✓ Benchmark complete")

### Create comparison table

In [ ]:
if torch.cuda.is_available():
    # Aggregate results
    summary = benchmark_df.groupby('precision')['generation_time_ms'].mean().reset_index()
    summary.columns = ['precision', 'avg_time_ms']

    # Add memory usage
    summary['memory_gb'] = summary['precision'].map(memory_usage)

    # Calculate relative to fp16
    fp16_time = summary[summary['precision'] == 'fp16']['avg_time_ms'].values[0]
    fp16_memory = summary[summary['precision'] == 'fp16']['memory_gb'].values[0]

    summary['speedup_vs_fp16'] = (fp16_time / summary['avg_time_ms']).round(2)
    summary['memory_saving_vs_fp16'] = ((1 - summary['memory_gb'] / fp16_memory) * 100).round(1)

    # Reorder columns for clarity
    summary = summary[['precision', 'avg_time_ms', 'speedup_vs_fp16', 'memory_gb', 'memory_saving_vs_fp16']]

    print("\n📊 Quantization Impact Summary:")
    print(summary.to_string(index=False))

    print("\n💡 Key Findings:")
    int8_speedup = summary[summary['precision'] == 'int8']['speedup_vs_fp16'].values[0]
    int8_memory = summary[summary['precision'] == 'int8']['memory_saving_vs_fp16'].values[0]
    int4_speedup = summary[summary['precision'] == 'int4']['speedup_vs_fp16'].values[0]
    int4_memory = summary[summary['precision'] == 'int4']['memory_saving_vs_fp16'].values[0]

    print(f"  int8: {int8_speedup:.1f}x speedup, {int8_memory:.0f}% memory savings")
    print(f"  int4: {int4_speedup:.1f}x speedup, {int4_memory:.0f}% memory savings")
    print(f"\n  Trade-off: Minor speedup, significant memory savings")
    print(f"  Quality: Outputs remain coherent (validate manually above)")

## Exercise 3 — Inference Profiling

**Goal**: Identify where inference time is spent in the forward pass.

**Method**: Use `torch.profiler` to break down time by operation type.

**Compare**: Attention vs FFN vs LayerNorm at different sequence lengths.

In [ ]:
if not torch.cuda.is_available():
    print("⚠ Skipping Exercise 3 — GPU required for profiling")
else:
    print("✓ GPU available — proceeding with inference profiling")

### Profile forward pass

In [ ]:
if torch.cuda.is_available():
    from torch.profiler import profile, ProfilerActivity, record_function

    # Load a fresh fp16 model for profiling
    print("Loading GPT-2 for profiling...")
    torch.cuda.empty_cache()
    profile_model = AutoModelForCausalLM.from_pretrained(
        "gpt2",
        torch_dtype=torch.float16,
        device_map="auto",
    )
    profile_tokenizer = AutoTokenizer.from_pretrained("gpt2")
    profile_tokenizer.pad_token = profile_tokenizer.eos_token

    def profile_inference(model, tokenizer, seq_len: int) -> dict:
        """
        Profile model forward pass and return time breakdown.

        Returns:
            dict with keys: attention_ms, ffn_ms, layer_norm_ms, other_ms
        """
        # Create input of specified length
        input_text = "test " * (seq_len // 2)  # Approximate target length
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=seq_len).to(model.device)

        # Warmup
        with torch.no_grad():
            _ = model(**inputs)

        # Profile
        with profile(
            activities=[ProfilerActivity.CUDA],
            record_shapes=False,
            with_stack=False,
        ) as prof:
            with torch.no_grad():
                _ = model(**inputs)

        # Analyze results
        events = prof.key_averages()

        attention_time = 0
        ffn_time = 0
        layer_norm_time = 0
        other_time = 0

        for event in events:
            name = event.key.lower()
            cuda_time = event.cuda_time_total / 1000  # Convert to ms

            # Categorize operations
            if 'attention' in name or 'attn' in name or 'bmm' in name:
                attention_time += cuda_time
            elif 'gelu' in name or 'mlp' in name or ('linear' in name and 'attn' not in name):
                ffn_time += cuda_time
            elif 'norm' in name:
                layer_norm_time += cuda_time
            else:
                other_time += cuda_time

        return {
            'attention_ms': round(attention_time, 2),
            'ffn_ms': round(ffn_time, 2),
            'layer_norm_ms': round(layer_norm_time, 2),
            'other_ms': round(other_time, 2),
        }

    print("✓ Profiling function ready")

### Compare across sequence lengths

In [ ]:
if torch.cuda.is_available():
    seq_lengths = [128, 512, 1024]  # Using 1024 instead of 2048 for GPT-2 context limit

    profile_results = []
    for seq_len in seq_lengths:
        print(f"\nProfiling seq_len={seq_len}...")
        breakdown = profile_inference(profile_model, profile_tokenizer, seq_len)
        breakdown['seq_len'] = seq_len
        profile_results.append(breakdown)
        print(f"  Attention: {breakdown['attention_ms']:.2f} ms")
        print(f"  FFN: {breakdown['ffn_ms']:.2f} ms")
        print(f"  LayerNorm: {breakdown['layer_norm_ms']:.2f} ms")
        print(f"  Other: {breakdown['other_ms']:.2f} ms")

    profile_df = pd.DataFrame(profile_results)

    # Calculate percentages
    profile_df['total_ms'] = profile_df[['attention_ms', 'ffn_ms', 'layer_norm_ms', 'other_ms']].sum(axis=1)
    profile_df['attention_pct'] = (profile_df['attention_ms'] / profile_df['total_ms'] * 100).round(1)
    profile_df['ffn_pct'] = (profile_df['ffn_ms'] / profile_df['total_ms'] * 100).round(1)
    profile_df['layer_norm_pct'] = (profile_df['layer_norm_ms'] / profile_df['total_ms'] * 100).round(1)
    profile_df['other_pct'] = (profile_df['other_ms'] / profile_df['total_ms'] * 100).round(1)

    print("\n📊 Profiling Results:")
    print(profile_df[['seq_len', 'attention_ms', 'ffn_ms', 'layer_norm_ms', 'total_ms']].to_string(index=False))
    print("\nPercentage Breakdown:")
    print(profile_df[['seq_len', 'attention_pct', 'ffn_pct', 'layer_norm_pct']].to_string(index=False))

### Visualize time breakdown

In [ ]:
if torch.cuda.is_available():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Absolute time breakdown
    x = range(len(profile_df))
    width = 0.6

    ax1.bar(x, profile_df['attention_ms'], width, label='Attention', color='#3498db')
    ax1.bar(x, profile_df['ffn_ms'], width, bottom=profile_df['attention_ms'], label='FFN', color='#e74c3c')
    ax1.bar(x, profile_df['layer_norm_ms'], width,
            bottom=profile_df['attention_ms'] + profile_df['ffn_ms'], label='LayerNorm', color='#2ecc71')
    ax1.bar(x, profile_df['other_ms'], width,
            bottom=profile_df['attention_ms'] + profile_df['ffn_ms'] + profile_df['layer_norm_ms'],
            label='Other', color='#95a5a6')

    ax1.set_xlabel('Sequence Length')
    ax1.set_ylabel('Time (ms)')
    ax1.set_title('Inference Time Breakdown (Absolute)')
    ax1.set_xticks(x)
    ax1.set_xticklabels(profile_df['seq_len'])
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Percentage breakdown
    ax2.bar(x, profile_df['attention_pct'], width, label='Attention', color='#3498db')
    ax2.bar(x, profile_df['ffn_pct'], width, bottom=profile_df['attention_pct'], label='FFN', color='#e74c3c')
    ax2.bar(x, profile_df['layer_norm_pct'], width,
            bottom=profile_df['attention_pct'] + profile_df['ffn_pct'], label='LayerNorm', color='#2ecc71')
    ax2.bar(x, profile_df['other_pct'], width,
            bottom=profile_df['attention_pct'] + profile_df['ffn_pct'] + profile_df['layer_norm_pct'],
            label='Other', color='#95a5a6')

    ax2.set_xlabel('Sequence Length')
    ax2.set_ylabel('Percentage (%)')
    ax2.set_title('Inference Time Breakdown (Relative)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(profile_df['seq_len'])
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n💡 Observation:")
    print(f"  Attention time grows with sequence length (O(n²) complexity)")
    print(f"  At seq_len=128: ~{profile_df.iloc[0]['attention_pct']:.0f}% attention")
    print(f"  At seq_len=1024: ~{profile_df.iloc[-1]['attention_pct']:.0f}% attention")
    print(f"  FFN time stays relatively constant (linear ops)")

## Summary & Key Findings

**Exercise 1 — VRAM Calculator**:
- Key insight: VRAM scales linearly with model size, sequence length impacts KV cache
- Which models fit on RTX 4090: All 7B/13B in int4, 7B in fp16 at short sequences

**Exercise 2 — Quantization Impact**:
- Memory savings (int8 vs fp16): ~50%
- Speed improvement (int8 vs fp16): Minor (1.0-1.2x typically)
- Quality degradation: Minimal for perceptual tasks

**Exercise 3 — Inference Profiling**:
- Dominant operation at short sequences: FFN and other ops
- Dominant operation at long sequences: Attention (quadratic growth)
- How does attention scale with seq_len: O(n²) — becomes bottleneck beyond ~512 tokens